## Executive Summary

**Purpose:** Turn the designed gears and parts into what a gear shop and a machine shop need: the gear data table for each drawing, tooth thickness limits as measurable span and over-ball dimensions, the accuracy class with its tolerance values, fits as numeric limits, heat treatment and case depth, and a tooth outline for prototypes.

**What it does:**

- `iso286_limits` gives the limits of a fit such as H7, G6, j5, n6 for a nominal size (ISO 286-2 tables).
- `iso1328_tolerances` computes the single pitch, cumulative pitch, profile, helix and runout tolerances of ISO 1328-1:2013 for a flank tolerance class.
- `backlash_allowances` splits the minimum backlash of both planetary meshes into tooth thickness allowances for sun, planet and ring.
- `gear_data_sheet` assembles the ISO 1340 / DIN 3966-1 data table of one gear, with span or over-ball limits computed from the thickness allowances.
- `gear_outline` and `write_dxf` give the closed tooth outline of a gear as points or as a DXF polyline, for wire EDM or laser-cut prototypes.

## How It Fits Into the Gear Design Workflow

**Upstream:** `gears.involute` (geometry and measurement formulas), `gears.layout` (the set), `gears.rating` (materials), `gears.components` (fits named on the parts).

**Downstream:** `gears.design.DesignPackage`, which writes the data sheets, the drawing and the report to files.

In [ ]:
#| default_exp gears.manufacturing

In [ ]:
#| export
from __future__ import annotations

import math
from dataclasses import dataclass

import numpy as np

from emachines.gears.involute import GearGeometry, inv, inv_inverse, span_teeth, span_measurement, ball_diameter, ball_measurement
from emachines.gears.layout import PlanetarySet
from emachines.gears.rating import GearMaterial

## ISO 286 Fits

Standard tolerance grades IT5–IT11 and the fundamental deviations used on these parts, for nominal sizes up to 500 mm (ISO 286-2 tables, in µm). A hole letter is upper case, a shaft letter lower case. `iso286_limits(d, "H7")` returns the upper and lower deviation in mm.

In [ ]:
#| export
_RANGES = [3, 6, 10, 18, 30, 50, 80, 120, 180, 250, 315, 400, 500]
_IT = {5: [4, 5, 6, 8, 9, 11, 13, 15, 18, 20, 23, 25, 27],
       6: [6, 8, 9, 11, 13, 16, 19, 22, 25, 29, 32, 36, 40],
       7: [10, 12, 15, 18, 21, 25, 30, 35, 40, 46, 52, 57, 63],
       8: [14, 18, 22, 27, 33, 39, 46, 54, 63, 72, 81, 89, 97],
       9: [25, 30, 36, 43, 52, 62, 74, 87, 100, 115, 130, 140, 155],
       10: [40, 48, 58, 70, 84, 100, 120, 140, 160, 185, 210, 230, 250],
       11: [60, 75, 90, 110, 130, 160, 190, 220, 250, 290, 320, 360, 400],
       12: [100, 120, 150, 180, 210, 250, 300, 350, 400, 460, 520, 570, 630]}
# fundamental deviations [µm]: g → es (negative), G → EI; k, m, n, p → ei
_FD = {"g": [-2, -4, -5, -6, -7, -9, -10, -12, -14, -15, -17, -18, -20],
       "k": [0, 1, 1, 1, 2, 2, 2, 3, 3, 4, 4, 4, 5],
       "m": [2, 4, 6, 7, 8, 9, 11, 13, 15, 17, 20, 21, 23],
       "n": [4, 8, 10, 12, 15, 17, 20, 23, 27, 31, 34, 37, 40],
       "p": [6, 12, 15, 18, 22, 26, 32, 37, 43, 50, 56, 62, 68]}
# N7 holes (ES) [µm]; EI = ES − IT7
_N7_ES = [-4, -4, -4, -5, -7, -8, -9, -10, -12, -14, -14, -16, -17]
# j5, j6 (es, ei) [µm]
_J = {5: [(2, -2), (3, -2), (4, -2), (5, -3), (5, -4), (6, -5), (6, -7), (6, -9), (7, -11), (7, -13), (7, -16), (7, -18), (7, -20)],
      6: [(4, -2), (6, -2), (7, -2), (8, -3), (9, -4), (11, -5), (12, -7), (13, -9), (14, -11), (16, -13), (16, -16), (18, -18), (20, -20)]}


def _idx(d: float) -> int:
    for i, hi in enumerate(_RANGES):
        if d <= hi:
            return i
    raise ValueError("ISO 286 tables here cover sizes up to 500 mm")


def it_grade(d: float, grade: int) -> float:
    "Standard tolerance IT`grade` [mm] for nominal size d [mm]."
    return _IT[grade][_idx(d)] / 1000


def iso286_limits(d: float, fit: str) -> tuple[float, float]:
    """
    (upper, lower) deviation [mm] of the tolerance class `fit` (e.g. 'H7', 'G6', 'N7', 'h11', 'js6', 'j5', 'k6', 'n6', 'p6', 'P9', 'JS9').
    """
    letter, grade = fit.rstrip("0123456789"), int(fit[len(fit.rstrip("0123456789")):])
    i = _idx(d)
    IT = _IT[grade][i] / 1000
    if letter == "H": return IT, 0.0
    if letter == "h": return 0.0, -IT
    if letter in ("js", "JS"): return IT / 2, -IT / 2
    if letter == "G": return -_FD["g"][i] / 1000 + IT, -_FD["g"][i] / 1000
    if letter == "g": return _FD["g"][i] / 1000, _FD["g"][i] / 1000 - IT
    if letter == "j":
        es_um, ei_um = _J[grade][i]
        return es_um / 1000, ei_um / 1000
    if letter in ("k", "m", "n", "p"):
        lower = 0.0 if (letter == "k" and grade > 7) else _FD[letter][i] / 1000
        return lower + IT, lower
    if letter == "N" and grade == 7:
        ES = _N7_ES[i] / 1000
        return ES, ES - IT
    if letter == "P" and grade >= 8:
        ES = -_FD["p"][i] / 1000
        return ES, ES - IT
    raise ValueError(f"tolerance class {fit} not in this table")


def fmt_fit(d: float, fit: str, digits: int = 3) -> str:
    "Drawing text such as 'Ø12 H7 (+0.018/0)'."
    up, lo = iso286_limits(d, fit)
    f = lambda v: "0" if abs(v) < 5e-7 else f"{v:+.{digits}f}"
    return f"Ø{d:g} {fit} ({f(up)}/{f(lo)})"

In [ ]:
print(fmt_fit(12, "H7"), fmt_fit(12, "n6"), fmt_fit(9, "G6"), fmt_fit(6, "j5"), fmt_fit(29.4, "h11"))
assert iso286_limits(12, "H7") == (0.018, 0.0) and iso286_limits(12, "n6") == (0.023, 0.012)
assert iso286_limits(25, "G6") == (0.020, 0.007) and iso286_limits(40, "P9") == (-0.026, -0.088)

## ISO 1328-1:2013 Flank Tolerances: Theory

The 2013 edition defines the tolerances of flank tolerance class $A$ (1 to 11; the old grades 5–8 map roughly to classes 5–8) by formulas in the reference diameter $d$, module $m$ and face width $b$ (mm, results in µm), each scaled by $(\sqrt2)^{A-5}$:

| Tolerance | Formula at class 5 |
|---|---|
| single pitch $f_{pT}$ | $0.001d + 0.4m + 5$ |
| cumulative pitch (total) $F_{pT}$ | $0.002d + 0.55\sqrt d + 0.7m + 12$ |
| total profile $F_{\alpha T}$ | $3.2\sqrt m + 0.22\sqrt d + 0.7$ |
| profile form $f_{f\alpha T}$ | $2.5\sqrt m + 0.17\sqrt d + 0.5$ |
| profile slope $f_{H\alpha T}$ | $2\sqrt m + 0.14\sqrt d + 0.5$ |
| total helix $F_{\beta T}$ | $0.07\sqrt d + 0.45\sqrt b + 4$ |
| helix form and slope $f_{f\beta T} = f_{H\beta T}$ | $0.05\sqrt d + 0.35\sqrt b + 4$ |
| radial runout $F_{rT}$ | $0.9\,F_{pT}$ (ISO 1328-2 guidance) |

Values above 10 µm are rounded to whole µm, values from 5 to 10 µm to 0.5 µm, and smaller values to 0.1 µm.

In [ ]:
#| export
def _round_tol(v: float) -> float:
    if v > 10: return float(round(v))
    if v >= 5: return round(v * 2) / 2
    return round(v, 1)


def iso1328_tolerances(g: GearGeometry, A: int) -> dict[str, float]:
    "ISO 1328-1:2013 flank tolerances [µm] of gear g for flank tolerance class A."
    d, m, b = g.d, g.m, g.b
    k = math.sqrt(2) ** (A - 5)
    F_p = (0.002 * d + 0.55 * math.sqrt(d) + 0.7 * m + 12) * k
    return {"f_pT": _round_tol((0.001 * d + 0.4 * m + 5) * k),
            "F_pT": _round_tol(F_p),
            "F_alphaT": _round_tol((3.2 * math.sqrt(m) + 0.22 * math.sqrt(d) + 0.7) * k),
            "f_falphaT": _round_tol((2.5 * math.sqrt(m) + 0.17 * math.sqrt(d) + 0.5) * k),
            "f_HalphaT": _round_tol((2.0 * math.sqrt(m) + 0.14 * math.sqrt(d) + 0.5) * k),
            "F_betaT": _round_tol((0.07 * math.sqrt(d) + 0.45 * math.sqrt(b) + 4) * k),
            "f_HbetaT": _round_tol((0.05 * math.sqrt(d) + 0.35 * math.sqrt(b) + 4) * k),
            "F_rT": _round_tol(0.9 * F_p)}

In [ ]:
t6 = iso1328_tolerances(GearGeometry(z=18, m=1.5, b=20), 6)
print(t6)
assert t6["F_pT"] > t6["f_pT"] and iso1328_tolerances(GearGeometry(z=18, m=1.5, b=20), 7)["F_pT"] > t6["F_pT"]

## Backlash and Tooth Thickness Allowances: Theory

Gears are made with their teeth slightly thinner than the theoretical value so that the mesh has backlash. The smallest normal backlash a mesh should have (ISO/TR 10064-2, also AGMA 2002) is

$$j_{bn,min} = \tfrac{2}{3}\,(0.06 + 0.0005\,a_w + 0.03\,m)\ \text{mm}.$$

A centre distance error $\Delta a$ changes the backlash by $2\Delta a\sin\alpha_w$: a shorter distance takes backlash away on an external mesh, a longer one on an internal mesh. With the centre distance held to $\pm f_a$ (here IT7/2 of $a_w$, the carrier pin-hole position tolerance), the thickness reductions of the two gears at the reference circle must add up to at least

$$S = \frac{j_{bn,min} + 2f_a\sin\alpha_w}{\cos\alpha}.$$

The planet is in both meshes, so it takes half of the larger requirement and the sun and ring make up the rest of their own mesh. This gives the *upper* thickness allowance $A_{sne}$ of each gear (negative: tooth thinner than nominal). The *lower* allowance is $A_{sni} = A_{sne} - T_{sn}$. The tolerance $T_{sn}$ must at least cover the thickness change caused by runout, $2F_{rT}\tan\alpha$ (DIN 3967 practice); 15 µm is added for the measuring and machining spread and the result rounded up to 5 µm. Agree $T_{sn}$ with your gear shop. A shop that grinds can often hold less, a shop that shapes the ring may need more.

The largest backlash follows from the lower allowances and the long side of the centre distance tolerance, and is reported so you can check it against the positioning accuracy your application needs.

In [ ]:
#| export
@dataclass(frozen=True)
class ThicknessAllowance:
    "Tooth thickness allowances of one gear on the reference circle [mm] (negative = thinner)."
    A_sne: float
    A_sni: float

    @property
    def T_sn(self) -> float: return self.A_sne - self.A_sni


def _round_up(v: float, step: float = 0.005) -> float:
    return math.ceil(v / step - 1e-9) * step


def backlash_allowances(ps: PlanetarySet, A: int) -> dict:
    """
    Thickness allowances of sun, planet and ring, plus min/max backlash of both meshes [mm].

    A: ISO 1328-1 flank tolerance class (sets the runout used for the thickness tolerance).
    """
    a_w = ps.a_w
    f_a = it_grade(a_w, 7) / 2
    m = ps.sun.m
    jmin = 2 / 3 * (0.06 + 0.0005 * a_w + 0.03 * m)
    ca = math.cos(ps.sun.alpha)
    S_sp = (jmin + 2 * f_a * math.sin(ps.sun_planet.alpha_w)) / ca
    S_pr = (jmin + 2 * f_a * math.sin(ps.planet_ring.alpha_w)) / ca
    planet_up = _round_up(max(S_sp, S_pr) / 2)
    ups = {"sun": _round_up(max(S_sp - planet_up, 0.25 * S_sp)), "planet": planet_up,
           "ring": _round_up(max(S_pr - planet_up, 0.25 * S_pr))}
    out = {}
    for key, g in (("sun", ps.sun), ("planet", ps.planet), ("ring", ps.ring)):
        Fr = iso1328_tolerances(g, A)["F_rT"] / 1000
        T = _round_up(2 * Fr * math.tan(g.alpha) + 0.015)
        out[key] = ThicknessAllowance(-ups[key], -ups[key] - T)
    j = {}
    for mesh, k1, k2, aw in (("sun–planet", "sun", "planet", ps.sun_planet.alpha_w),
                             ("planet–ring", "planet", "ring", ps.planet_ring.alpha_w)):
        jn_min = -(out[k1].A_sne + out[k2].A_sne) * ca - 2 * f_a * math.sin(aw)
        jn_max = -(out[k1].A_sni + out[k2].A_sni) * ca + 2 * f_a * math.sin(aw)
        j[mesh] = (jn_min, jn_max)
    return {"allowances": out, "backlash_n": j, "j_bn_min_required": jmin, "f_a": f_a}

## Gear Data Sheet: Theory

ISO 1340 and DIN 3966-1 list what a gear drawing must carry so that the gear can be cut and inspected without the designer's calculation. `gear_data_sheet` produces those rows, plus the heat treatment and surface data.

**Inspection limits.** The span $W_k$ (external gears) and the dimension over balls $M_d$ are evaluated at the two thickness limits $s + A_{sne}$ and $s + A_{sni}$ with the formulas of `gears.involute`. They are the dimensions the gear inspector measures.

**Tip diameter tolerance.** External gears h11, internal gears H11 (the tip diameter only needs to be held loosely, because it only affects the contact ratio and the clearances checked in `gears.layout`).

**Case depth.** For carburized gears the effective case depth (CHD to 550 HV, ISO 2639) after grinding is chosen in proportion to the module: $0.15\,m$ to $0.2\,m$, at least 0.3 mm. That depth is deep enough to carry the subsurface shear stress under the contact, and shallow enough that the tooth tip does not through-harden. Specify the depth before grinding as this plus the grinding stock (about 0.05–0.1 mm per flank). For nitrided gears, the nitriding hardness depth (NHD) is 0.3–0.5 mm.

In [ ]:
#| export
def case_depth(g: GearGeometry, mat: GearMaterial) -> str:
    "Recommended hardening depth text for the drawing."
    if mat.kind == "case":
        lo, hi = max(0.3, 0.15 * g.m), max(0.45, 0.2 * g.m)
        return f"CHD 550 HV1 = {lo:.2f}–{hi:.2f} mm after grinding (ISO 2639)"
    if mat.kind == "nitrided":
        return "NHD (core + 50 HV) = 0.30–0.50 mm, compound layer ≤ 10 µm"
    return "through hardened — no case depth"


@dataclass(frozen=True)
class DataRow:
    "One row of a gear data table."
    item: str
    symbol: str
    value: str


def gear_data_sheet(g: GearGeometry, mate: GearGeometry, a_w: float, allowance: ThicknessAllowance,
                    A: int, mat: GearMaterial, finish: str, n_mate_label: str = "",
                    d_active: float | None = None) -> list[DataRow]:
    """
    ISO 1340 / DIN 3966-1 data table rows for gear g meshing with `mate` at centre distance a_w.

    finish: 'ground', 'hobbed' or 'shaped' (sets roughness text and process route).
    d_active: start of active profile (from `gears.involute.mesh_check`) for the form-diameter requirement.
    """
    deg = math.degrees
    s_up, s_lo = g.s + allowance.A_sne, g.s + allowance.A_sni
    tol = iso1328_tolerances(g, A)
    rows = [
        DataRow("Gear", "", g.name),
        DataRow("Type", "", "spur, internal" if g.internal else "spur, external"),
        DataRow("Number of teeth", "z", f"{g.z}"),
        DataRow("Normal module", "m_n", f"{g.m:g} mm"),
        DataRow("Pressure angle", "α_n", f"{deg(g.alpha):g}°"),
        DataRow("Helix angle", "β", "0°"),
        DataRow("Basic rack profile", "", f"ISO 53 profile A: h_aP = {g.ha:g} m, h_fP = {g.hf:g} m, ρ_fP = {g.rho_f:g} m"),
        DataRow("Profile shift coefficient", "x", f"{g.x:+.4f}"),
        DataRow("Reference diameter", "d", f"{g.d:.4f} mm"),
        DataRow("Base diameter", "d_b", f"{g.db:.4f} mm"),
        DataRow("Tip diameter", "d_a", f"{g.da:.3f} mm " + ("H11 " + _dev(g.da, 'H11') if g.internal else "h11 " + _dev(g.da, 'h11'))),
        DataRow("Root diameter", "d_f", f"{g.df:.3f} mm " + ("(min)" if g.internal else "(max)")),
        DataRow("Root form diameter", "d_Ff", (f"≥ {g.d_form:.3f} mm" if g.internal else f"≤ {g.d_form:.3f} mm")
                + (f" (active profile starts at {d_active:.3f} mm)" if d_active else "")),
        DataRow("Face width", "b", f"{g.b:g} mm ±0.1"),
        DataRow("Tooth thickness on d (arc)", "s", f"{g.s:.4f} mm, allowances {allowance.A_sne:+.3f}/{allowance.A_sni:+.3f} mm"),
    ]
    if not g.internal:
        k = span_teeth(g)
        Wu, Wl = span_measurement(g, s_up, k), span_measurement(g, s_lo, k)
        rows.append(DataRow("Span measurement", f"W_k (k = {k})", f"{Wu:.4f} / {Wl:.4f} mm"))
    D = ball_diameter(g)
    Mu, dc = ball_measurement(g, D, s_up)
    Ml, _ = ball_measurement(g, D, s_lo)
    lab = "Dimension between balls" if g.internal else "Dimension over balls"
    hi, lo = (Ml, Mu) if g.internal else (Mu, Ml)
    rows.append(DataRow(lab, f"M_d (D_M = {D:.2f} mm)", f"{hi:.4f} / {lo:.4f} mm"
                         + (" (odd z: two-ball across one tooth space)" if g.z % 2 else "")))
    rows += [
        DataRow("Flank tolerance class", "", f"ISO 1328-1:2013 class {A}"),
        DataRow("Single pitch tolerance", "f_pT", f"{tol['f_pT']:g} µm"),
        DataRow("Total cumulative pitch tolerance", "F_pT", f"{tol['F_pT']:g} µm"),
        DataRow("Total profile tolerance", "F_αT", f"{tol['F_alphaT']:g} µm"),
        DataRow("Profile slope tolerance", "f_HαT", f"±{tol['f_HalphaT']:g} µm"),
        DataRow("Total helix tolerance", "F_βT", f"{tol['F_betaT']:g} µm"),
        DataRow("Helix slope tolerance", "f_HβT", f"±{tol['f_HbetaT']:g} µm (crown 3–5 µm recommended on planets)" if g.name == "planet" else f"±{tol['f_HbetaT']:g} µm"),
        DataRow("Radial runout tolerance", "F_rT", f"{tol['F_rT']:g} µm"),
        DataRow("Mating gear", "", f"{mate.name}, z = {mate.z}{n_mate_label}"),
        DataRow("Centre distance", "a_w", f"{a_w:.4f} mm ±{it_grade(a_w, 7) / 2 * 1000:.0f} µm"),
        DataRow("Material", "", mat.name),
        DataRow("Heat treatment", "", mat.treatment),
        DataRow("Hardness", "", mat.hardness),
        DataRow("Hardening depth", "", case_depth(g, mat)),
        DataRow("Flank roughness", "", {"ground": "Ra ≤ 0.4 µm (Rz ≤ 3)", "hobbed": "Ra ≤ 1.6 µm", "shaped": "Ra ≤ 1.6 µm"}[finish]),
        DataRow("Root roughness", "", "Ra ≤ 3.2 µm, no grinding notch in the root"),
        DataRow("Tip edge", "", f"chamfer or round 0.1–0.15 m ({0.1 * g.m:.2f}–{0.15 * g.m:.2f} mm)"),
        DataRow("Process route", "", _route(g, mat, finish)),
    ]
    return rows


def _dev(d: float, fit: str) -> str:
    up, lo = iso286_limits(d, fit)
    f = lambda v: "0" if abs(v) < 5e-7 else f"{v:+.3f}"
    return f"({f(up)}/{f(lo)})"


def _route(g: GearGeometry, mat: GearMaterial, finish: str) -> str:
    cut = "shape (pinion-type cutter) or power-skive" if g.internal else "hob"
    if mat.kind == "case":
        ht = "carburize, harden, temper"
    elif mat.kind == "nitrided":
        ht = "gas nitride (after finish cutting)"
    else:
        ht = ""
    steps = ["turn blank (bore and faces as datum)", cut]
    if mat.kind == "nitrided":
        steps += [ht]
    elif ht:
        steps += [ht, "grind bore and faces", "grind flanks" if finish == "ground" else "hone or lap flanks"]
    elif finish == "ground":
        steps += ["grind flanks"]
    return " → ".join(steps)

### Example

Data sheets of the 18/27/72 set, flank class 6:

In [ ]:
from emachines.gears.layout import PlanetarySet
from emachines.gears.rating import MATERIALS
from emachines.gears.involute import mesh_check
ps = PlanetarySet.build(18, 27, 72, m=1.5, n_planets=3, b=20)
bl = backlash_allowances(ps, 6)
print({k: (round(v.A_sne, 3), round(v.A_sni, 3)) for k, v in bl["allowances"].items()},
      {k: tuple(round(x, 3) for x in v) for k, v in bl["backlash_n"].items()}, f"j_bn,min = {bl['j_bn_min_required']:.3f}")
sheet = gear_data_sheet(ps.sun, ps.planet, ps.a_w, bl["allowances"]["sun"], 6, MATERIALS["case_carburized"], "ground",
                        d_active=ps.sun_planet.d_Nf1)
for r in sheet:
    print(f"{r.item:34s} {r.symbol:20s} {r.value}")
for mesh, (jmin, jmax) in bl["backlash_n"].items():
    assert jmin >= bl["j_bn_min_required"] - 1e-9 and jmax > jmin

In [ ]:
ring_sheet = gear_data_sheet(ps.ring, ps.planet, ps.a_w, bl["allowances"]["ring"], 7, MATERIALS["qt_nitrided"], "shaped",
                             d_active=ps.planet_ring.d_Nf2)
for r in ring_sheet[10:17]:
    print(f"{r.item:34s} {r.symbol:20s} {r.value}")

## Tooth Outline for Prototypes

`gear_outline` returns the closed outline of the whole gear as points. On an external gear the flanks are exact involutes from the tip down to the root form circle, and the root fillet below them is generated the way a hob makes it: as the envelope of the ISO 53 rack's tip radius rolling on the reference circle. On an internal gear the fillet depends on the shaper cutter, so the outline runs straight from the form circle to the root circle. The outline is right for wire EDM or laser-cut prototypes, test fixtures and drawings; production gears are generated by the hob or shaper from the data sheet. The thickness allowance can be applied (default: the mean of the two limits). `write_dxf` writes the outline as an R12 DXF polyline, in mm.

In [ ]:
#| export
def _involute_xy(rb: float, r: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    "Polar angle of the involute at radius r (from the start on the base circle)."
    a = np.arccos(np.clip(rb / r, -1, 1))
    return r, np.tan(a) - a


def _rack_fillet(g: GearGeometry, ds: float, n_bins: int = 60, n_phi: int = 1200) -> tuple[np.ndarray, np.ndarray]:
    """
    Root fillet and root of a rack-generated (hobbed) external gear, as polar points (angle from the
    space centre, radius) across the part of the tooth space below the root form circle.
    The fillet is the envelope of the tool tip radius ρ_fP as the rack rolls on the reference circle.
    """
    m, a = g.m, g.alpha
    r = g.d / 2
    rho = g.rho_f * m
    y_ref = r + g.x * m                          # tool reference line (shifted by x·m)
    y_tip = y_ref - g.hf * m                     # tool tip line = gear root circle
    y_c = y_tip + rho
    half_ref = np.pi * m / 4 - ds / 2            # thinner gear tooth ↔ wider tool tooth
    u_c = half_ref - (y_ref - y_c) * np.tan(a) - rho / np.cos(a)
    t = np.linspace(-np.pi / 2, -a, 40)
    pu = np.concatenate([u_c + rho * np.cos(t), np.linspace(0, max(u_c, 0), 8)])
    py = np.concatenate([y_c + rho * np.sin(t), np.full(8, y_tip)])
    pu = np.concatenate([pu, -pu]); py = np.concatenate([py, py])
    phi = np.linspace(-3 * np.pi / g.z, 3 * np.pi / g.z, n_phi)[:, None]
    X = pu[None, :] - r * phi
    xg = X * np.cos(phi) + py[None, :] * np.sin(phi)
    yg = -X * np.sin(phi) + py[None, :] * np.cos(phi)
    th = (np.arctan2(yg, xg) - np.pi / 2).ravel()
    rr = np.hypot(xg, yg).ravel()
    r_form = max(g.d_form / 2, g.db / 2)
    rb = g.db / 2
    half_form = (g.s + ds) / g.d + float(inv(g.alpha)) - float(inv(np.arccos(min(1.0, rb / r_form))))
    lim = np.pi / g.z - half_form                # space half-angle at the form circle
    edges = np.linspace(-lim, lim, n_bins + 1)
    idx = np.digitize(th, edges) - 1
    ok = (idx >= 0) & (idx < n_bins)
    rmin = np.full(n_bins, np.inf)
    np.minimum.at(rmin, idx[ok], rr[ok])
    centres = (edges[:-1] + edges[1:]) / 2
    good = np.isfinite(rmin)
    rmin = np.interp(centres, centres[good], rmin[good])
    rmin = np.minimum(rmin, r_form)
    ang = np.concatenate([[-lim], centres, [lim]])
    rad = np.concatenate([[r_form], rmin, [r_form]])
    return ang, rad


def gear_outline(g: GearGeometry, ds: float | None = None, n_flank: int = 24, n_arc: int = 6) -> np.ndarray:
    """
    Closed outline (N×2 array, mm) of gear g, centred on the origin, one tooth on the +x axis.

    External gears: exact involute flanks from the root form circle to the tip, and the root fillet
    generated by the ISO 53 rack (tool tip radius ρ_fP) below it. Internal gears: involute flanks and
    a straight run from the form circle to the root circle (the real fillet depends on the shaper cutter).
    ds: thickness change on the reference circle [mm] (negative: thinner; default 0).
    """
    ds = 0.0 if ds is None else ds
    s = g.s + ds
    rb = g.db / 2
    ra, rf = g.da / 2, g.df / 2
    r_form = max(g.d_form / 2, rb) if not g.internal else min(g.d_form / 2, rf)
    psi_ref = s / g.d                                  # half tooth angle on the reference circle
    inv_a = float(inv(g.alpha))
    pitch = 2 * np.pi / g.z
    if g.internal:
        r = np.linspace(ra, r_form, n_flank)           # from tip (inside) outward
        _, ang = _involute_xy(rb, np.maximum(r, rb))
        half = psi_ref - inv_a + ang                   # internal tooth widens outward
    else:
        r = np.linspace(r_form, ra, n_flank)
        _, ang = _involute_xy(rb, r)
        half = psi_ref + inv_a - ang
        f_ang, f_rad = _rack_fillet(g, ds)
    pts = []
    for k in range(g.z):
        c = k * pitch
        if g.internal:
            # r runs tip (inner) → form (outer). Right flank root→tip, tip arc across the tooth,
            # left flank tip→root, root arc across the space to the next tooth.
            right = [(rf, c - half[-1])] + [(ri, c - h) for ri, h in zip(r[::-1], half[::-1])]
            tip = [(ra, t) for t in np.linspace(c - half[0], c + half[0], n_arc)[1:-1]]
            left = [(ri, c + h) for ri, h in zip(r, half)] + [(rf, c + half[-1])]
            nxt = (k + 1) * pitch - half[-1]
            seq = right + tip + left + [(rf, t) for t in np.linspace(c + half[-1], nxt, n_arc)[1:-1]]
        else:
            # right flank form→tip, tip arc, left flank tip→form, generated root across the next space
            right = [(ri, c - h) for ri, h in zip(r, half)]
            tip = [(ra, t) for t in np.linspace(c - half[-1], c + half[-1], n_arc)[1:-1]]
            left = [(ri, c + h) for ri, h in zip(r[::-1], half[::-1])]
            root = [(ri, c + pitch / 2 + ti) for ti, ri in zip(f_ang[1:-1], f_rad[1:-1])]
            seq = right + tip + left + root
        pts += seq
    P = np.array([(ri * np.cos(t), ri * np.sin(t)) for ri, t in pts])
    return P


def write_dxf(path: str, outlines: list[np.ndarray], circles: tuple | list = ()) -> None:
    "Write closed polylines (and circles: (x, y, r)) to an R12 ASCII DXF file, units mm."
    L = ["0", "SECTION", "2", "HEADER", "9", "$INSUNITS", "70", "4", "0", "ENDSEC",
         "0", "SECTION", "2", "ENTITIES"]
    for P in outlines:
        L += ["0", "POLYLINE", "8", "0", "66", "1", "70", "1"]
        for x, y in P:
            L += ["0", "VERTEX", "8", "0", "10", f"{x:.5f}", "20", f"{y:.5f}"]
        L += ["0", "SEQEND"]
    for x, y, r in circles:
        L += ["0", "CIRCLE", "8", "0", "10", f"{x:.5f}", "20", f"{y:.5f}", "40", f"{r:.5f}"]
    L += ["0", "ENDSEC", "0", "EOF"]
    with open(path, "w") as f:
        f.write("\n".join(L) + "\n")

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(6, 6))
P = gear_outline(ps.ring, -0.05)
ax.plot(*P.T, lw=0.8, label="ring")
rho = np.pi * ps.planet.z / ps.ring.z                     # ring phase so its teeth mesh with planet 1
P = P @ np.array([[np.cos(rho), np.sin(rho)], [-np.sin(rho), np.cos(rho)]])
ax.lines[0].set_data(*P.T)
for k in range(3):
    th = 2 * np.pi * k / 3
    psi = th * (1 + ps.sun.z / ps.planet.z) + np.pi - np.pi / ps.planet.z   # planet phase that meshes with the sun
    R_ = np.array([[np.cos(psi), np.sin(psi)], [-np.sin(psi), np.cos(psi)]])
    Pp = gear_outline(ps.planet, -0.05) @ R_ + ps.a_w * np.array([np.cos(th), np.sin(th)])
    ax.plot(*Pp.T, lw=0.8, color="C1")
Ps = gear_outline(ps.sun, -0.05)
ax.plot(*Ps.T, lw=0.8, color="C2")
ax.set_aspect("equal"); ax.set_axis_off()
plt.tight_layout()

Checks on the outline: every point lies between the root and tip circles, and the tooth thickness on the reference circle matches $s$:

In [ ]:
for g in (ps.sun, ps.planet, ps.ring):
    P = gear_outline(g)
    rr = np.hypot(P[:, 0], P[:, 1])
    lo, hi = min(g.da, g.df) / 2, max(g.da, g.df) / 2
    assert rr.min() >= lo - 1e-6 and rr.max() <= hi + 1e-6, g.name
    # tooth thickness on the reference circle, interpolated on the first tooth's two flanks
    th = np.arctan2(P[:, 1], P[:, 0])
    flank = (np.abs(th) < np.pi / g.z) & (rr > lo + 1e-6) & (rr < hi - 1e-6)
    sides = []
    for sgn in (-1, 1):
        sel = flank & (np.sign(th) == sgn)
        order = np.argsort(rr[sel])
        sides.append(np.interp(g.d / 2, rr[sel][order], th[sel][order]))
    s_meas = (sides[1] - sides[0]) * g.d / 2
    print(f"{g.name:6s} thickness on d from the outline {s_meas:.4f} mm, nominal s = {g.s:.4f} mm")
    assert abs(s_meas - g.s) < 1e-3, g.name

---

## Tests

In [ ]:
#| hide
import tempfile, os
# ISO 286 spot checks against the printed tables
assert iso286_limits(30, "H7") == (0.021, 0.0) and iso286_limits(30.5, "H7") == (0.025, 0.0)
assert iso286_limits(8, "j5") == (0.004, -0.002) and iso286_limits(20, "k6") == (0.015, 0.002)
assert iso286_limits(50, "h11") == (0.0, -0.16) and iso286_limits(12, "N7") == (-0.005, -0.023)
assert iso286_limits(14, "P9") == (-0.018, -0.061) and iso286_limits(14, "JS9") == (0.0215, -0.0215)
# ISO 1328 class steps by √2
t5 = iso1328_tolerances(GearGeometry(z=40, m=2, b=20), 5)
t7 = iso1328_tolerances(GearGeometry(z=40, m=2, b=20), 7)
assert abs(t7["F_pT"] / t5["F_pT"] - 2) < 0.1
# span limits bracket and differ by T_sn·cos α
al = bl["allowances"]["sun"]
k = span_teeth(ps.sun)
dW = span_measurement(ps.sun, ps.sun.s + al.A_sne, k) - span_measurement(ps.sun, ps.sun.s + al.A_sni, k)
assert math.isclose(dW, al.T_sn * math.cos(ps.sun.alpha))
# internal gear: thinner teeth → larger dimension between balls (first value of the row is the upper limit)
row = next(r for r in ring_sheet if r.item.startswith("Dimension between"))
hi_, lo_ = [float(x) for x in row.value.split(" mm")[0].split(" / ")]
assert hi_ > lo_
# DXF writes and closes
with tempfile.TemporaryDirectory() as td:
    pth = os.path.join(td, "sun.dxf")
    write_dxf(pth, [gear_outline(ps.sun)], [(0, 0, 7.0)])
    txt = open(pth).read()
    assert txt.count("VERTEX") == len(gear_outline(ps.sun)) and txt.strip().endswith("EOF")

## References

- ISO 1328-1:2013 — *Cylindrical gears — ISO system of flank tolerance classification — Part 1*.
- ISO/TR 10064-2:1996 — *Code of inspection practice — Radial composite deviations, runout, tooth thickness and backlash*.
- ISO 1340:1976 and DIN 3966-1:1978 — *Information to be given on drawings of involute gears*.
- DIN 3967:1978 — *Backlash, tooth thickness allowances, tooth thickness tolerances*.
- ISO 286-2:2010 — *Tables of standard tolerance classes and limit deviations for holes and shafts*.
- ISO 2639:2002 — *Steels — Determination and verification of the depth of carburized and hardened cases*.